In [0]:
import dlt
import pandas as pd
from pyspark.sql.functions import col, expr
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-5909396472671535>, line 1
----> 1 import dlt
      2 import pandas as pd
      3 from pyspark.sql.functions import col, expr

File /databricks/python_shell/lib/dbruntime/autoreload/discoverability/autoreload_discoverability_hook.py:96, in AutoreloadDiscoverabilityHook._patched_import(self, name, *args, **kwargs)
     90 if not self._should_hint and (
     91     (module := sys.modules.get(absolute_name)) is not None and
     92     (fname := get_allowed_file_name_or_none(module)) is not None and
     93     (mtime := os.stat(fname).st_mtime) > self.last_mtime_by_modname.get(
     94         absolute_name, float("inf")) and not self._should_hint):
     95     self._should_hint = True
---> 96 module = self._original_builtins_import(name, *args, **kwargs)
     97 if (fname := fname or get_allowed_file_name_or_none(module)) is

In [0]:
# 1. Simulate hardcoded data using a static DataFrame
# DLT Must return DATAFRAME from the function
@dlt.table(
    name="raw_transactions"
)
def load_raw_transactions():
    data = [
        ("t1", "u1", 100.0, "US", "2025-04-01T10:00:00"),
        ("t2", "u2", -50.0, "UK", "2025-04-01T10:15:00"),   # invalid amount
        ("t3", "u3", 200.0, "IN", "2025-04-01T11:00:00"),
        ("t4", None, 150.0, "US", "2025-04-01T11:30:00"),   # missing user_id
        ("t5", "u5", 120.0, "CA", "2025-04-01T12:00:00")
    ]
    
    schema = StructType([
        StructField("transaction_id", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("amount", DoubleType(), True),
        StructField("country", StringType(), True),
        StructField("timestamp", StringType(), True)
    ])
    
    spark_df = spark.createDataFrame(data, schema)
    return spark_df.withColumn("timestamp", col("timestamp").cast(TimestampType()))

Name,Type
transaction_id,string
user_id,string
amount,double
country,string
timestamp,timestamp


In [0]:
@dlt.table(
    name="validated_transactions"
)
@dlt.expect_or_drop("valid_amount", "amount > 0") # DATA QUALITY
@dlt.expect("user_id_present", "user_id IS NOT NULL") # DATA QUALITY
def validated_transactions():
    # raw_transactions is upstream for validated_transactions , raw_transactions is given as name in previous dlt
    return dlt.read("raw_transactions") # CRREATING PIPELINE/CHAIN, dtd.read return DF

Name,Type
transaction_id,string
user_id,string
amount,double
country,string
timestamp,timestamp


In [0]:
@dlt.view(
    name="invalid_transactions"
)
def invalid_transactions():
    df = dlt.read("raw_transactions")
    return df.filter((col("amount") <= 0) | (col("user_id").isNull()))

Name,Type
transaction_id,string
user_id,string
amount,double
country,string
timestamp,timestamp


In [0]:
@dlt.view(
    name="final_transactions_view"
)
def final_transactions():
    df = dlt.read("validated_transactions")
    return df.select("transaction_id", "user_id", "amount", "country", "timestamp")

Name,Type
transaction_id,string
user_id,string
amount,double
country,string
timestamp,timestamp
